In [1]:
"""
CBO transparency demo (toy):
- Reads OBBA/P.L. 119-21 Medicaid supplemental PDF from disk (no web download).
- Extracts Section 71119 "Community Engagement Requirements" block and key headline numbers.
- Builds a toy HISIM2-inspired mapping from:
    (procedural burden as a Medicaid ASC shift) + (responsiveness scaling beta) + (choice-set restrictions)
  to:
    uninsured among Medicaid leavers, implied uninsured level in 2034, and a welfare sign.
- Uses a certainty-equivalent (CE) / compensating-variation-like welfare wedge rather than the earlier
  unstable "million-dollar CV" calculation.

This is NOT HISIM2 and does NOT reproduce CBO scoring. It is an auditable illustration of:
text -> parameters -> outputs -> decision, and how "in-the-weeds" modeling choices can flip welfare sign.
"""

import re
from pathlib import Path

import numpy as np
import pandas as pd
import pdfplumber


# ============================================================
# 0) CONFIG
# ============================================================

# Put the PDFs in your working directory (or adjust paths here)
OBBA_PDF = Path("PL-119-21-Medicaid _0.pdf")     # your local OBBA supplemental
HISIM_PDF = Path("57205-HISIM.pdf")              # optional local HISIM2 doc (can be missing)

# Decision / welfare parameters (toy)
RAMP_EQUIV_YEARS = 7.0     # 2027:1/3, 2028:2/3, 2029-2034:1 -> 7 "full-effect years"
N_subject_2034_m = 15.0    # denominator used to convert "lose Medicaid in 2034" level into post-policy Medicaid share
                           # choose and report what this means (subject/enrolled/eligible in 2034 absent policy).

# Toy health spending states (stand-in for HISIM2's 14 health states)
RNG_SEED = 0
S = 14
CONSUMPTION_Y = 35000.0

# Toy uninsured financial exposure parameters (the "gory" knobs)
UNINS_COLLECTION_RATE = 0.2
MEDICAID_OOP_RATE = 0.05
ALPHA_BAD_DEBT = 0.25       # converts uninsured bad debt into effective resource cost (toy analog of "theta(w)" channel)
UNINS_CAP = 2000.0          # uninsured OOP cap / charity care / non-collection intensity knob (toy)

# Discrete-choice model knobs
# - deltas are ASCs (ease/access/awareness/stigma)
# - Delta_ASC_Medicaid is the "procedural burden" shift from work requirements
DELTAS_BASE = {"Medicaid": 2.0, "Uninsured": 0.0, "ESI": 0.5, "Nongroup": 0.2}
CHOICE_SET_FULL = ("Medicaid", "Uninsured", "ESI", "Nongroup")
CHOICE_SET_RESTRICTED = ("Medicaid", "Uninsured")  # extreme restriction to illustrate the channel

# Responsiveness scaling (toy analog of HISIM2 scaling factor beta)
BETA_SCALES = [0.5, 1.0, 2.0, 4.0]


# ============================================================
# 1) PDF UTILITIES
# ============================================================

def pdf_to_text(pdf_path, max_pages=None):
    parts = []
    with pdfplumber.open(pdf_path) as pdf:
        pages = pdf.pages[:max_pages] if max_pages else pdf.pages
        for p in pages:
            parts.append(p.extract_text() or "")
    return "\n".join(parts)

def find_block(text, start_pat, end_pat=None, max_len=25000):
    m = re.search(start_pat, text, flags=re.IGNORECASE)
    if not m:
        return None
    start = m.start()
    if end_pat:
        n = re.search(end_pat, text[start:], flags=re.IGNORECASE)
        end = start + n.start() if n else min(len(text), start + max_len)
    else:
        end = min(len(text), start + max_len)
    return text[start:end]

def extract_float(pattern, s):
    m = re.search(pattern, s, flags=re.IGNORECASE | re.DOTALL)
    return float(m.group(1)) if m else None


# ============================================================
# 2) EXTRACT OBBA 71119 NUMBERS
# ============================================================

assert OBBA_PDF.exists(), f"Missing OBBA PDF: {OBBA_PDF.resolve()}"

obba_text = pdf_to_text(OBBA_PDF)

block_71119 = find_block(
    obba_text,
    start_pat=r"Community Engagement Requirements\.\s*Section\s*71119",
    end_pat=r"Other Medicaid-Related Effects|Uncertainty"
)

if block_71119 is None:
    raise ValueError("Could not locate the 71119 block. Try adjusting the start/end regex.")

N_fail_2034 = extract_float(r"about\s+(\d+(\.\d+)?)\s+million\s+in\s+2034.*?will lose\s+coverage", block_71119)
N_admin_2034 = extract_float(r"about\s+(\d+(\.\d+)?)\s+million\s+additional.*?lose Medicaid coverage in\s+2034", block_71119)
p_keep_pct = extract_float(r"about\s+(\d+(\.\d+)?)\s+percent.*?will keep their Medicaid\s+coverage", block_71119)
uninsured_2034 = extract_float(r"increase by\s+(\d+(\.\d+)?)\s+million\s+in\s+2034", block_71119)
deficit_red_B = extract_float(r"decrease deficits by\s+\$?(\d+(\.\d+)?)\s+billion", block_71119)

if None in [N_fail_2034, N_admin_2034, p_keep_pct, uninsured_2034, deficit_red_B]:
    print("WARNING: Some OBBA numbers were not parsed. Parsed values:")
    print(N_fail_2034, N_admin_2034, p_keep_pct, uninsured_2034, deficit_red_B)

p_keep = p_keep_pct / 100.0
N_loss_total_2034 = (1 - p_keep) * N_fail_2034 + N_admin_2034
uninsured_share_among_losers_obba = uninsured_2034 / N_loss_total_2034

print("\nExtracted OBBA 71119:")
print(f"  N_fail_2034 (m): {N_fail_2034}")
print(f"  N_admin_2034 (m): {N_admin_2034}")
print(f"  p_keep: {p_keep}")
print(f"  total lose Medicaid 2034 (m): {N_loss_total_2034}")
print(f"  uninsured increase 2034 (m): {uninsured_2034}")
print(f"  uninsured share among losers: {uninsured_share_among_losers_obba}")
print(f"  deficit reduction ($B): {deficit_red_B}")

# break-even coverage value threshold (the main transparency object)
uninsured_personyears = uninsured_2034 * 1e6 * RAMP_EQUIV_YEARS
CV_star = (deficit_red_B * 1e9) / uninsured_personyears
print(f"\nImplied break-even value threshold CV* ($/uninsured person-year): {CV_star:,.2f}")


# ============================================================
# 3) TOY HISIM-LIKE "VALUE OF MEDICAID" VIA CERTAINTY EQUIVALENT
# ============================================================

rng = np.random.default_rng(RNG_SEED)
p_state = np.ones(S) / S
H = np.sort(rng.lognormal(mean=8.0, sigma=0.7, size=S))  # synthetic annual spending states

def CE_log(consumption_by_state, p_state):
    """Certainty equivalent under log utility: CE = exp(E[log c])."""
    consumption_by_state = np.maximum(consumption_by_state, 1.0)
    return float(np.exp(np.sum(p_state * np.log(consumption_by_state))))

def build_costs(H, medicaid_oop_rate=0.05, unins_cap=2000.0, unins_collection_rate=0.2):
    """
    Medicaid: oop = rate * H
    Uninsured: oop = min(H, cap) + collection_rate * max(0, H-cap)
              baddebt = (1-collection_rate) * max(0, H-cap)
    """
    oop_med = medicaid_oop_rate * H
    oop_unins = np.minimum(H, unins_cap) + unins_collection_rate * np.maximum(0, H - unins_cap)
    baddebt = (1 - unins_collection_rate) * np.maximum(0, H - unins_cap)
    return oop_med, oop_unins, baddebt

def value_of_medicaid_CV(consumption=35000.0,
                         medicaid_oop_rate=0.05,
                         unins_cap=2000.0,
                         unins_collection_rate=0.2,
                         alpha_bad_debt=0.25):
    """
    A stable, interpretable toy 'value of Medicaid' per person-year:
      CV = CE_medicaid - CE_uninsured

    Uninsured net consumption includes an effective dollar penalty alpha * baddebt.
    (Toy analog of uninsured financial exposure + bad-debt channel described in HISIM2.)
    """
    oop_med, oop_unins, baddebt = build_costs(H, medicaid_oop_rate, unins_cap, unins_collection_rate)

    c_med = consumption - oop_med
    c_un  = consumption - oop_unins - alpha_bad_debt * baddebt

    CE_med = CE_log(c_med, p_state)
    CE_un  = CE_log(c_un, p_state)
    return CE_med - CE_un

def net_welfare(deficit_reduction_B, uninsured_m_2034, value_per_personyear, ramp_years=7.0):
    uninsured_py = uninsured_m_2034 * 1e6 * ramp_years
    return deficit_reduction_B * 1e9 - value_per_personyear * uninsured_py

cv_base = value_of_medicaid_CV(consumption=CONSUMPTION_Y,
                               unins_cap=UNINS_CAP,
                               unins_collection_rate=UNINS_COLLECTION_RATE,
                               alpha_bad_debt=ALPHA_BAD_DEBT,
                               medicaid_oop_rate=MEDICAID_OOP_RATE)
W_base = net_welfare(deficit_red_B, uninsured_2034, cv_base, RAMP_EQUIV_YEARS)

print("\nToy welfare wedge (CE-based):")
print(f"  value_of_medicaid (CV) $/py: {cv_base:,.2f}")
print(f"  net welfare ($B): {W_base/1e9:,.2f}")
print(f"  policy good? (net welfare >=0): {W_base >= 0}")


# ============================================================
# 4) DISCRETE CHOICE: ASC SHIFT + BETA + CHOICE SET RESTRICTION
# ============================================================

def softmax(u):
    u = np.array(u, dtype=float)
    u = u - u.max()
    e = np.exp(u)
    return e / e.sum()

def V_alt(consumption, alt,
          unins_cap=2000.0,
          unins_collection_rate=0.2,
          alpha_bad_debt=0.25,
          medicaid_oop_rate=0.05):
    """
    Deterministic value index for each alternative. We keep it simple but consistent with the CE-based welfare story:
    - Medicaid and Uninsured: use expected log net consumption with uninsured bad debt penalty alpha*baddebt.
    - ESI and Nongroup: assign simple premium + oop shares (toy); no bad debt.
    """
    oop_med, oop_unins, baddebt = build_costs(H, medicaid_oop_rate, unins_cap, unins_collection_rate)

    if alt == "Medicaid":
        net = consumption - oop_med
        return float(np.sum(p_state * np.log(np.maximum(net, 1.0))))
    if alt == "Uninsured":
        net = consumption - oop_unins - alpha_bad_debt * baddebt
        return float(np.sum(p_state * np.log(np.maximum(net, 1.0))))
    if alt == "ESI":
        prem = 2500.0
        oop = 0.15 * H
        net = consumption - prem - oop
        return float(np.mean(np.log(np.maximum(net, 1.0))))
    if alt == "Nongroup":
        prem = 4500.0
        oop = 0.20 * H
        net = consumption - prem - oop
        return float(np.mean(np.log(np.maximum(net, 1.0))))
    raise ValueError(alt)

def choice_probs(consumption, beta_scale, deltas, choice_set, **kwargs):
    Us = []
    for alt in choice_set:
        Us.append(beta_scale * V_alt(consumption, alt, **kwargs) + deltas.get(alt, 0.0))
    p = softmax(Us)
    return dict(zip(choice_set, p))

def calibrate_medicaid_ASC_shift(target_P_medicaid_post,
                                 consumption,
                                 beta_scale,
                                 deltas_base,
                                 choice_set,
                                 lo=-150.0,
                                 hi=0.0,
                                 **kwargs):
    """
    Find Delta (<=0) such that P_post(Medicaid) = target_P_medicaid_post.
    """
    def Pmed(Delta):
        d = dict(deltas_base)
        d["Medicaid"] = deltas_base["Medicaid"] + Delta
        return choice_probs(consumption, beta_scale, d, choice_set, **kwargs)["Medicaid"]

    f_lo = Pmed(lo) - target_P_medicaid_post
    f_hi = Pmed(hi) - target_P_medicaid_post
    # If not bracketing, return endpoint that is closer
    if f_lo * f_hi > 0:
        return lo if abs(f_lo) < abs(f_hi) else hi

    a, b = lo, hi
    for _ in range(300):
        mid = 0.5 * (a + b)
        f_mid = Pmed(mid) - target_P_medicaid_post
        if abs(f_mid) < 1e-10:
            return mid
        # if Pmed(mid) > target, need more negative Delta -> move upper bound down
        if f_mid > 0:
            b = mid
        else:
            a = mid
    return 0.5 * (a + b)

def uninsured_among_leavers(p_post):
    leave = 1 - p_post.get("Medicaid", 0.0)
    if leave <= 1e-15:
        return 0.0
    return p_post.get("Uninsured", 0.0) / leave

# Convert OBBA "lose Medicaid in 2034" into target post-policy Medicaid share (given N_subject choice).
target_P_medicaid_post = 1.0 - (N_loss_total_2034 / N_subject_2034_m)
print(f"\nCalibration target using N_subject_2034_m={N_subject_2034_m}:")
print(f"  target P_post(Medicaid) = {target_P_medicaid_post:.6f}")

# ---- (i) Calibrate ESI/Nongroup ASCs so baseline mapping matches OBBA uninsured share among leavers ----
# We do this at beta_scale=1.0 in the FULL choice set by shifting ESI and Nongroup ASCs down together.
def calibrate_substitution_mapping(target_uninsured_among_leavers,
                                   consumption,
                                   beta_scale,
                                   deltas_base,
                                   choice_set,
                                   **kwargs):
    """
    Choose a common shift t (<=0) applied to ESI and Nongroup ASCs so that,
    after also calibrating Medicaid Delta to hit target P_post(Medicaid),
    uninsured among leavers matches the OBBA mapping.
    """
    lo, hi = -30.0, 0.0  # search range for ESI/Nongroup accessibility reduction

    def g(t):
        deltas = dict(deltas_base)
        if "ESI" in deltas: deltas["ESI"] += t
        if "Nongroup" in deltas: deltas["Nongroup"] += t

        Delta = calibrate_medicaid_ASC_shift(target_P_medicaid_post,
                                             consumption=consumption,
                                             beta_scale=beta_scale,
                                             deltas_base=deltas,
                                             choice_set=choice_set,
                                             **kwargs)
        deltas_post = dict(deltas)
        deltas_post["Medicaid"] += Delta
        p_post = choice_probs(consumption, beta_scale, deltas_post, choice_set, **kwargs)
        return uninsured_among_leavers(p_post) - target_uninsured_among_leavers

    g_lo, g_hi = g(lo), g(hi)
    if g_lo * g_hi > 0:
        # no bracket; return the closer endpoint
        return lo if abs(g_lo) < abs(g_hi) else hi

    a, b = lo, hi
    for _ in range(200):
        mid = 0.5 * (a + b)
        gm = g(mid)
        if abs(gm) < 1e-8:
            return mid
        if gm > 0:
            # too much uninsured among leavers -> make ESI/Nongroup MORE attractive (t less negative)
            a = mid
        else:
            b = mid
    return 0.5 * (a + b)

sub_shift = calibrate_substitution_mapping(
    target_uninsured_among_leavers=uninsured_share_among_losers_obba,
    consumption=CONSUMPTION_Y,
    beta_scale=1.0,
    deltas_base=DELTAS_BASE,
    choice_set=CHOICE_SET_FULL,
    unins_cap=UNINS_CAP,
    unins_collection_rate=UNINS_COLLECTION_RATE,
    alpha_bad_debt=ALPHA_BAD_DEBT,
    medicaid_oop_rate=MEDICAID_OOP_RATE
)

DELTAS_CAL = dict(DELTAS_BASE)
DELTAS_CAL["ESI"] += sub_shift
DELTAS_CAL["Nongroup"] += sub_shift

print("\nCalibrated substitution accessibility shift (applied to ESI and Nongroup ASCs) to match OBBA mapping:")
print(f"  sub_shift = {sub_shift:.6f}")
print("  deltas_calibrated:", DELTAS_CAL)

# ---- (ii) Main table: vary beta_scale and choice set; always re-calibrate Medicaid Delta to hit target P_post(Medicaid) ----
rows = []
for beta_scale in BETA_SCALES:
    for cs_name, choice_set in [("full", CHOICE_SET_FULL), ("restricted", CHOICE_SET_RESTRICTED)]:
        deltas_use = dict(DELTAS_CAL)
        # if restricted, drop ASCs for options not in choice set
        deltas_use = {k: v for k, v in deltas_use.items() if k in choice_set}

        Delta = calibrate_medicaid_ASC_shift(
            target_P_medicaid_post=target_P_medicaid_post,
            consumption=CONSUMPTION_Y,
            beta_scale=beta_scale,
            deltas_base=deltas_use,
            choice_set=choice_set,
            unins_cap=UNINS_CAP,
            unins_collection_rate=UNINS_COLLECTION_RATE,
            alpha_bad_debt=ALPHA_BAD_DEBT,
            medicaid_oop_rate=MEDICAID_OOP_RATE
        )

        deltas_post = dict(deltas_use)
        deltas_post["Medicaid"] = deltas_use["Medicaid"] + Delta

        p_post = choice_probs(
            CONSUMPTION_Y, beta_scale, deltas_post, choice_set,
            unins_cap=UNINS_CAP,
            unins_collection_rate=UNINS_COLLECTION_RATE,
            alpha_bad_debt=ALPHA_BAD_DEBT,
            medicaid_oop_rate=MEDICAID_OOP_RATE
        )

        ual = uninsured_among_leavers(p_post)

        # Map the model-implied uninsured-among-leavers into an implied 2034 uninsured level,
        # holding fixed the OBBA-implied "lose Medicaid in 2034" level N_loss_total_2034.
        uninsured_pred_2034_m = ual * N_loss_total_2034

        # Welfare (toy): deficit reduction - (CV per py) * uninsured person-years
        cv = value_of_medicaid_CV(consumption=CONSUMPTION_Y,
                                  unins_cap=UNINS_CAP,
                                  unins_collection_rate=UNINS_COLLECTION_RATE,
                                  alpha_bad_debt=ALPHA_BAD_DEBT,
                                  medicaid_oop_rate=MEDICAID_OOP_RATE)

        W = net_welfare(deficit_red_B, uninsured_pred_2034_m, cv, RAMP_EQUIV_YEARS)

        rows.append({
            "beta_scale": beta_scale,
            "choice_set": cs_name,
            "Delta_ASC_Medicaid": Delta,
            "P_post_Medicaid": p_post.get("Medicaid", np.nan),
            "uninsured_among_leavers": ual,
            "uninsured_2034_pred_m": uninsured_pred_2034_m,
            "CV_value_$per_py": cv,
            "net_welfare_$B": W/1e9,
            "policy_good?": W >= 0
        })

df = pd.DataFrame(rows)
print("\n=== ASC shift + beta sensitivity + choice-set restriction (toy) ===")
print(df.to_string(index=False))


# ============================================================
# 5) OPTIONAL: quick HISIM2 sentence tagging (if local HISIM PDF exists)
# ============================================================

if HISIM_PDF.exists():
    hisim_text = pdf_to_text(HISIM_PDF, max_pages=50)  # keep it light; adjust pages if desired

    def split_sentences(s):
        s = re.sub(r"\s+", " ", s).strip()
        return re.split(r"(?<=[\.\?\!])\s+", s)

    hisim_sents = [x for x in split_sentences(hisim_text) if len(x) > 80]

    # lightweight keyword-based tagging (no external model downloads)
    def keyword_tag(sent):
        t = sent.lower()
        if "alternative-specific constant" in t or "alternative specific constant" in t:
            return "ASC (access/awareness/stigma)"
        if "restricted the choice sets" in t or "choice sets" in t and "restricted" in t:
            return "choice-set restriction"
        if "uninsured people are not fully exposed" in t or "charity care" in t or "never collected" in t or "bankruptcy" in t:
            return "uninsured financial exposure"
        if "scaling factor" in t and "responsiveness" in t:
            return "scaling beta / responsiveness"
        if "calibration" in t and ("targets" in t or "minimize" in t):
            return "calibration targets"
        return None

    tagged = []
    for s in hisim_sents:
        tag = keyword_tag(s)
        if tag:
            tagged.append({"tag": tag, "sentence": s})

    df_tag = pd.DataFrame(tagged)
    print("\n=== Example HISIM2 sentences (keyword-tagged) ===")
    if len(df_tag) == 0:
        print("No hits found on first 50 pages; increase max_pages or adjust keywords.")
    else:
        # show a few per category
        for tag in df_tag["tag"].unique():
            print(f"\n[{tag}]")
            for sent in df_tag[df_tag["tag"] == tag]["sentence"].head(3).tolist():
                print("-", sent)
else:
    print("\n(HISIM PDF not found locally; skipping sentence tagging.)")


Extracted OBBA 71119:
  N_fail_2034 (m): 2.9
  N_admin_2034 (m): 2.8
  p_keep: 0.05
  total lose Medicaid 2034 (m): 5.555
  uninsured increase 2034 (m): 5.3
  uninsured share among losers: 0.9540954095409541
  deficit reduction ($B): 317.0

Implied break-even value threshold CV* ($/uninsured person-year): 8,544.47

Toy welfare wedge (CE-based):
  value_of_medicaid (CV) $/py: 2,210.21
  net welfare ($B): 235.00
  policy good? (net welfare >=0): True

Calibration target using N_subject_2034_m=15.0:
  target P_post(Medicaid) = 0.629667

Calibrated substitution accessibility shift (applied to ESI and Nongroup ASCs) to match OBBA mapping:
  sub_shift = -4.040705
  deltas_calibrated: {'Medicaid': 2.0, 'Uninsured': 0.0, 'ESI': -3.540704518556595, 'Nongroup': -3.8407045185565947}

=== ASC shift + beta sensitivity + choice-set restriction (toy) ===
 beta_scale choice_set  Delta_ASC_Medicaid  P_post_Medicaid  uninsured_among_leavers  uninsured_2034_pred_m  CV_value_$per_py  net_welfare_$B  poli